In [37]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from trading_system.pipelines import single_ticker_long_short as ann
from trading_system.pipelines import single_ticker_long_short_features as annF
from trading_system.backtest import lib as v3

pd.set_option('display.max_columns', 240)
pd.set_option('display.width', 260)
np.random.seed(1)


In [ ]:
# ------------------------------
# Inputs
# ------------------------------
DATASET_PATH = PROJECT_ROOT / 'data' / 'processed' / 'cac40_daily.parquet'

TICKER = 'SAF.PA'
PRICE_COL = 'adj_close'

# Mets ici ton DataFrame custom si besoin (sinon laisse None)
CUSTOM_DF = None
# Mets ici ton DataFrame déjà labélisé (doit contenir Label_id). Sinon laisse None
CUSTOM_LABELS_DF = None

# Si CUSTOM_LABELS_DF est None, on applique la labelisation ANN
LABEL_WINDOW = 30

# ------------------------------
# ANN hyperparams
# ------------------------------
ANN_PARAMS = dict(
    epochs=500,
    alpha=1e-3,
    hidden=32,
    do_dropout=False,
    dropout_percent=0.1,
    batch_size=32,
    train_ratio=0.7,
    val_ratio=0.15,
    context_len=20,
    early_stopping_patience=50,
    early_stopping_min_delta=1e-4,
)

# ------------------------------
# Backtest config
# ------------------------------
RUN_EXTENDED = True
PERSIST_RUN = True

BT_OVERRIDES = dict(
    symbol=TICKER,
    timeframe='1d',
    fees_bps=5.0,
    slippage_bps=0.0,
    stop_loss_pct=0.12,
    take_profit_pct=0.04,
    notes='ANN_long_short -> backtest_lib external pipeline',
)


In [39]:
def ensure_training_schema(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    if 'date' not in out.columns:
        if isinstance(out.index, pd.DatetimeIndex):
            out = out.reset_index().rename(columns={'index': 'date'})
        else:
            raise ValueError("Le DataFrame doit avoir une colonne 'date' ou un DatetimeIndex.")

    out['date'] = pd.to_datetime(out['date'], utc=True, errors='coerce')
    if out['date'].isna().any():
        raise ValueError("Timestamps invalides dans la colonne 'date'.")

    if 'adj_close' not in out.columns and 'close' in out.columns:
        out['adj_close'] = out['close']

    required = ['open', 'high', 'low', 'close', 'adj_close', 'volume']
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(f"Colonnes manquantes pour ANN: {missing}")

    for c in required:
        out[c] = pd.to_numeric(out[c], errors='coerce')
    if out[required].isna().any().any():
        raise ValueError("NaN detectes dans les colonnes OHLCV requises.")

    return out.sort_values('date').reset_index(drop=True)

if CUSTOM_DF is not None:
    df_raw = CUSTOM_DF.copy()
else:
    df_raw = ann.read_parquet_dataset(DATASET_PATH)
    if 'ticker' in df_raw.columns:
        df_raw = df_raw[df_raw['ticker'] == TICKER].copy()

df_raw = ensure_training_schema(df_raw)

if CUSTOM_LABELS_DF is not None:
    train_df = ensure_training_schema(CUSTOM_LABELS_DF.copy())
    if 'Label_id' not in train_df.columns:
        raise ValueError("CUSTOM_LABELS_DF doit contenir la colonne 'Label_id'.")
    train_df['Label_id'] = pd.to_numeric(train_df['Label_id'], errors='coerce').astype(int)
    if 'Label' not in train_df.columns:
        label_map = {0: 'Sell', 1: 'Hold', 2: 'Buy'}
        train_df['Label'] = train_df['Label_id'].map(label_map)
    label_stats = train_df['Label'].value_counts(dropna=False).to_dict()
else:
    train_df, label_stats = ann.labelling(df_raw.copy(), LABEL_WINDOW, price_col=PRICE_COL)

print('Data rows:', len(train_df))
print('Label stats:', label_stats)
display(train_df.head(5))


Data rows: 6728
Label stats: {'Buy': 75, 'Hold': 6577, 'Sell': 76}


,date,ticker,company,open,high,low,close,adj_close,volume,dividends,stock_splits,Label,Label_id
0,2000-01-03 00:00:00+00:00,SAF.PA,Safran,46.340000,46.633999,45.962002,46.004002,31.136086,516675.0,0.0,0.0,Hold,1
1,2000-01-04 00:00:00+00:00,SAF.PA,Safran,46.340000,46.396000,42.195999,44.001999,29.781103,1036500.0,0.0,0.0,Hold,1
2,2000-01-05 00:00:00+00:00,SAF.PA,Safran,42.672001,42.672001,39.998001,40.698002,27.544918,1354725.0,0.0,0.0,Hold,1
3,2000-01-06 00:00:00+00:00,SAF.PA,Safran,40.334000,42.993999,40.334000,42.237999,28.587204,630975.0,0.0,0.0,Hold,1
4,2000-01-07 00:00:00+00:00,SAF.PA,Safran,42.938000,44.268002,41.467999,43.330002,29.326292,2045100.0,0.0,0.0,Hold,1


In [40]:
best, test_metrics, benchmark = ann.train_one_trial(train_df, **ANN_PARAMS)

print('Test metrics:')
display(pd.DataFrame([test_metrics]))
print('Simple ANN benchmark comparison:')
display(pd.DataFrame([benchmark]))

market_df = best['advanced_backtest_market']
labels_df = best['advanced_backtest_labels']

print('Advanced market rows:', len(market_df))
display(market_df.head(3))
display(labels_df.head(3))



train rows: 4272 | val rows: 915 | test rows: 916
X_train: (4253, 720)
X_val: (915, 720)
X_test: (916, 720)
context_len: 20 | feature_dim: 36
W0: (720, 32)
b0: (1, 32)
W1: (32, 3)
b1: (1, 3)
class_weights: [29.534721  0.341031 29.534721]

Starting training ===========================================

epoch 1/500 
| loss = 1.0907          | acc(train) = 0.977     | acc(val) = 0.975       | bal_acc(val) = 0.333    | macro_f1(val) = 0.329 
| precision_buy = 0.000  | precision_sell = 0.000 | precision_hold = 0.975 
| recall_buy = 0.000     | recall_sell = 0.000    | recall_hold = 1.000

epoch 2/500 
| loss = 1.0730          | acc(train) = 0.977     | acc(val) = 0.975       | bal_acc(val) = 0.333    | macro_f1(val) = 0.329 
| precision_buy = 0.000  | precision_sell = 0.000 | precision_hold = 0.975 
| recall_buy = 0.000     | recall_sell = 0.000    | recall_hold = 1.000

epoch 3/500 
| loss = 1.0575          | acc(train) = 0.977     | acc(val) = 0.975       | bal_acc(val) = 0.333    | macro

,acc,bal_acc,macro_f1,precision_sell,recall_sell,precision_hold,recall_hold,precision_buy,recall_buy
0,0.959607,0.491815,0.475807,0.117647,0.2,0.983127,0.975446,0.3,0.3


Simple ANN benchmark comparison:


,initial_capital,model_final_capital,buy_hold_final_capital,model_pnl,buy_hold_pnl,outperformance
0,10000.0,14172.343395,30064.128424,4172.343395,20064.128424,-15891.78503


Advanced market rows: 916


,date,ticker,company,open,high,low,close,adj_close,volume,dividends,stock_splits,Label,Label_id,ret_2,ret_3,ret_5,ret_10,ret_20,ret_60,ret_120,dist_ma_5,dist_ma_20,dist_ma_60,ma_ratio_5_20,ma_ratio_20_60,bollinger_z_20,dist_high_20,dist_low_20,drawdown_20,drawdown_60,vol_5,vol_20,vol_60,vol_ratio_5_20,vol_ratio_20_60,atr_norm,parkinson_vol_20,garman_klass_vol_20,rogers_satchell_vol_20,dollar_volume,amihud_illiq,volume_z_20,ret_x_volume_shock,ret_x_illiq,autocorr_5,autocorr_20,variance_ratio_20,trend_regime,vol_regime
date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2022-08-09 00:00:00+00:00,2022-08-09 00:00:00+00:00,SAF.PA,Safran,107.239998,108.400002,106.720001,107.980003,104.676247,501404.0,0.0,0.0,Hold,1,0.018103,0.006150,0.007840,0.043890,0.065732,0.100417,-0.043344,0.006412,0.029705,0.096900,0.023144,0.065257,1.436551,-0.040372,0.056909,0.0,0.0,0.008445,0.009635,0.021171,0.876506,0.455075,0.043845,0.013758,0.015230,0.016591,5.248509e+07,1.170154e-10,-0.854735,-0.005096,6.977010e-13,-0.115996,-0.083952,0.278640,1.0,-1.0
2022-08-10 00:00:00+00:00,2022-08-10 00:00:00+00:00,SAF.PA,Safran,107.800003,109.940002,107.639999,109.820000,106.459946,519434.0,0.0,0.0,Hold,1,0.023104,0.035452,0.019117,0.056571,0.087112,0.127284,-0.005654,0.019646,0.042875,0.113254,0.022781,0.067486,2.019833,-0.031654,0.069734,0.0,0.0,0.010481,0.009967,0.021229,1.051513,0.469523,0.043883,0.013495,0.014678,0.015903,5.529892e+07,1.299440e-10,-0.697964,-0.011893,2.214265e-12,0.057275,0.009060,0.266921,1.0,-1.0
2022-08-11 00:00:00+00:00,2022-08-11 00:00:00+00:00,SAF.PA,Safran,110.000000,110.919998,109.160004,109.900002,106.537498,442460.0,0.0,0.0,Hold,1,0.017781,0.023850,0.024040,0.042497,0.091144,0.151990,-0.024096,0.015524,0.039106,0.111343,0.023222,0.069518,1.889664,-0.039510,0.064736,0.0,0.0,0.009919,0.009867,0.021019,1.005240,0.469431,0.043454,0.013272,0.014395,0.015641,4.713858e+07,1.284044e-10,-1.092327,-0.000796,9.353864e-14,-0.000758,-0.039533,0.268998,1.0,-1.0


,target_position,action,model_label_id
date,,,
2022-08-09 00:00:00+00:00,0,hold,1
2022-08-10 00:00:00+00:00,0,hold,1
2022-08-11 00:00:00+00:00,0,hold,1


In [41]:
start_iso = market_df.index.min().isoformat().replace('+00:00', 'Z')
end_iso = market_df.index.max().isoformat().replace('+00:00', 'Z')

cfg = v3.BacktestConfig(start=start_iso, end=end_iso)
cfg = replace(cfg, **BT_OVERRIDES)
v3.validate_config(v3.resolve_config(cfg))

run_obj, run_path = v3.execute_first_check_pipeline_external(
    cfg,
    market_df=market_df,
    labels_df=labels_df,
    persist=PERSIST_RUN,
    render=False,
)

print('Run path:', run_path)
display(pd.DataFrame([run_obj['core_metrics']]))
display(run_obj['trades'].head(20))
v3.render_viz_bundle(run_obj['viz_bundle'])


Run path: /Users/loic/Documents/Code/DL/Artificial-Neural-Network-based-Stock-Trading-System/research/runs/20260412T201533Z_saf.pa_1d


,initial_capital,final_capital,total_pnl,cumulative_return,sharpe_ratio,max_drawdown,win_rate,profit_factor,expectancy,total_trades,gain_loss_ratio,avg_trade_duration_hours,exposure
0,10000.0,15315.048752,5315.048752,0.531505,3.765341,-0.347221,0.770833,1.482775,110.730182,48,0.440825,608.5,0.934498


,trade_id,side,direction,entry_time,exit_time,entry_price,exit_price,entry_price_base,exit_price_base,entry_reason,exit_reason,equity_before_entry,entry_equity,gross_exit_equity,exit_equity,entry_fee,exit_fee,total_fees,net_pnl,return_pct,duration_bars,duration_hours,is_winner
0,1,long,1,2022-09-02 00:00:00+00:00,2022-09-09 00:00:00+00:00,100.080002,104.083202,100.080002,104.083202,signal_entry,take_profit,10000.000000,9995.000000,10394.800000,10389.602600,5.000000,5.197400,10.197400,389.602600,0.038960,6,168.0,True
1,2,long,1,2022-09-12 00:00:00+00:00,2022-09-26 00:00:00+00:00,104.480003,91.942403,104.480003,91.942403,signal_entry,stop_loss,10389.602600,10384.407799,9138.278863,9133.709723,5.194801,4.569139,9.763941,-1255.892877,-0.120880,11,336.0,False
2,3,long,1,2022-09-27 00:00:00+00:00,2022-10-04 00:00:00+00:00,93.279999,97.011199,93.279999,97.011199,signal_entry,take_profit,9133.709723,9129.142869,9494.308583,9489.561429,4.566855,4.747154,9.314009,355.851706,0.038960,6,168.0,True
3,4,long,1,2022-10-05 00:00:00+00:00,2022-10-13 00:00:00+00:00,98.669998,102.616798,98.669998,102.616798,signal_entry,take_profit,9489.561429,9484.816648,9864.209314,9859.277210,4.744781,4.932105,9.676885,369.715781,0.038960,7,192.0,True
4,5,long,1,2022-10-14 00:00:00+00:00,2022-10-19 00:00:00+00:00,104.040001,108.201601,104.040001,108.201601,signal_entry,take_profit,9859.277210,9854.347571,10248.521474,10243.397213,4.929639,5.124261,10.053899,384.120003,0.038960,4,120.0,True
5,6,long,1,2022-10-20 00:00:00+00:00,2022-10-24 00:00:00+00:00,105.660004,109.886404,105.660004,109.886404,signal_entry,take_profit,10243.397213,10238.275514,10647.806535,10642.482632,5.121699,5.323903,10.445602,399.085419,0.038960,3,96.0,True
6,7,long,1,2022-10-25 00:00:00+00:00,2022-10-28 00:00:00+00:00,109.459999,113.838399,109.459999,113.838399,signal_entry,take_profit,10642.482632,10637.161390,11062.647846,11057.116522,5.321241,5.531324,10.852565,414.633890,0.038960,4,72.0,True
7,8,long,1,2022-10-31 00:00:00+00:00,2022-11-23 00:00:00+00:00,112.000000,116.480000,112.000000,116.480000,signal_entry,take_profit,11057.116522,11051.587964,11493.651482,11487.904657,5.528558,5.746826,11.275384,430.788135,0.038960,18,552.0,True
8,9,long,1,2022-11-24 00:00:00+00:00,2023-01-06 00:00:00+00:00,116.080002,120.723202,116.080002,120.723202,signal_entry,take_profit,11487.904657,11482.160704,11941.447133,11935.476409,5.743952,5.970724,11.714676,447.571752,0.038960,31,1032.0,True
9,10,long,1,2023-01-09 00:00:00+00:00,2023-01-13 00:00:00+00:00,122.540001,127.441601,122.540001,127.441601,signal_entry,take_profit,11935.476409,11929.508671,12406.689018,12400.485673,5.967738,6.203345,12.171083,465.009264,0.038960,5,96.0,True


[viz] kpi_table


[viz] equity_curve


[viz] drawdown


[viz] pnl_distribution


[viz] heatmap_hour_day


[viz] heatmap_month_year


In [42]:
if RUN_EXTENDED:
    events_csv = Path('events_demo.csv')
    events_arg = events_csv if events_csv.exists() else None

    ext_obj, ext_path = v3.execute_complementary_pipeline(
        run_obj,
        cfg,
        events_csv_path=events_arg,
        persist=PERSIST_RUN,
        render=True,
    )

    ext_report = v3.run_extended_acceptance_checks(ext_obj, cfg, run_path=ext_path)
    display(ext_report)
    print('Extended path:', ext_path)

catalog = v3.load_run_catalog(cfg)
display(catalog.tail(10))


/Users/loic/Documents/Code/DL/Artificial-Neural-Network-based-Stock-Trading-System/ANN/backtest_lib.py:1847: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out['regime_state'] = out['regime_state'].ffill().bfill().fillna(0).astype(int)
/Users/loic/Documents/Code/DL/Artificial-Neural-Network-based-Stock-Trading-System/ANN/backtest_lib.py:2261: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out_full['anomaly_flag'] = out_full['anomaly_flag'].fillna(False).astype(bool)
/Users/loic/Documents/Code/DL/Artificial-Neural-Network-based-Stock-Trading-System/ANN/back

,initial_capital,final_capital,total_pnl,cumulative_return,sharpe_ratio,max_drawdown,win_rate,profit_factor,expectancy,total_trades,gain_loss_ratio,avg_trade_duration_hours,exposure
0,10000.0,15315.048752,5315.048752,0.531505,3.765341,-0.347221,0.770833,1.482775,110.730182,48,0.440825,608.5,0.934498


,split,enabled,strategy_final_capital,strategy_cumulative_return,benchmark_final_capital,benchmark_cumulative_return,benchmark_sharpe_ratio,benchmark_max_drawdown,alpha_vs_benchmark
0,train,True,15315.048752,0.531505,29144.286203,1.914429,8.264966,-0.230917,-1.382924


,split,initial_capital,final_capital,total_pnl,cumulative_return,sharpe_ratio,max_drawdown,win_rate,profit_factor,expectancy,total_trades,gain_loss_ratio,avg_trade_duration_hours,exposure,benchmark__strategy_final_capital,benchmark__strategy_cumulative_return,benchmark__benchmark_final_capital,benchmark__benchmark_cumulative_return,benchmark__benchmark_sharpe_ratio,benchmark__benchmark_max_drawdown,benchmark__alpha_vs_benchmark
0,train,10000.0,16372.320527,6372.320527,0.637232,4.248768,-0.18175,0.732143,1.404861,113.791438,56,0.513974,531.428571,0.94869,16372.320527,0.637232,29144.286203,1.914429,8.264966,-0.230917,-1.277197


,trade_id,side,direction,entry_time,exit_time,entry_price,exit_price,entry_price_base,exit_price_base,entry_reason,exit_reason,equity_before_entry,entry_equity,gross_exit_equity,exit_equity,entry_fee,exit_fee,total_fees,net_pnl,return_pct,duration_bars,duration_hours,is_winner,entry_hour,entry_day_name,entry_day_of_week,entry_ema_spread,entry_vol_24h,entry_ret_24h,entry_close,exit_close,market_move_pct,entry_position_context,entry_regime,event_count_near_trade,event_types_near_trade,winner_loser,anomaly_count_near_trade,anomaly_max_score,anomaly_top_feature,has_anomaly_near_trade
0,1,long,1,2022-09-02 00:00:00+00:00,2022-09-09 00:00:00+00:00,100.080002,104.083202,100.080002,104.083202,signal_entry,take_profit,10000.000000,9995.000000,10394.800000,10389.602600,5.000000,5.197400,10.197400,389.602600,0.03896,6,168.0,True,0,Friday,4,NaN,0.013961,NaN,102.360001,103.800003,0.014068,1,range,0,,winner,0,NaN,,False
1,2,long,1,2022-09-12 00:00:00+00:00,2022-09-26 00:00:00+00:00,104.480003,91.942403,104.480003,91.942403,signal_entry,stop_loss,10389.602600,10384.407799,9138.278863,9133.709723,5.194801,4.569139,9.763941,-1255.892877,-0.12088,11,336.0,False,0,Monday,0,NaN,0.014241,-0.019078,105.919998,92.889999,-0.123017,1,range,0,,loser,0,NaN,,False
2,3,long,1,2022-09-27 00:00:00+00:00,2022-10-04 00:00:00+00:00,93.279999,97.011199,93.279999,97.011199,signal_entry,take_profit,9133.709723,9129.142869,9494.308583,9489.561429,4.566855,4.747154,9.314009,355.851706,0.03896,6,168.0,True,0,Tuesday,1,NaN,0.016332,-0.146089,91.709999,99.150002,0.081125,1,range,0,,winner,1,0.047939,abs_ret_1h,True
3,4,long,1,2022-10-05 00:00:00+00:00,2022-10-13 00:00:00+00:00,98.669998,102.616798,98.669998,102.616798,signal_entry,take_profit,9489.561429,9484.816648,9864.209314,9859.277210,4.744781,4.932105,9.676885,369.715781,0.03896,7,192.0,True,0,Wednesday,2,NaN,0.019558,0.003619,99.830002,102.040001,0.022138,1,range,0,,winner,1,0.092797,body_pct,True
4,5,long,1,2022-10-14 00:00:00+00:00,2022-10-19 00:00:00+00:00,104.040001,108.201601,104.040001,108.201601,signal_entry,take_profit,9859.277210,9854.347571,10248.521474,10243.397213,4.929639,5.124261,10.053899,384.120003,0.03896,4,120.0,True,0,Friday,4,NaN,0.020349,-0.034743,102.239998,106.040001,0.037167,1,range,0,,winner,0,NaN,,False
5,6,long,1,2022-10-20 00:00:00+00:00,2022-10-24 00:00:00+00:00,105.660004,109.886404,105.660004,109.886404,signal_entry,take_profit,10243.397213,10238.275514,10647.806535,10642.482632,5.121699,5.323903,10.445602,399.085419,0.03896,3,96.0,True,0,Thursday,3,NaN,0.019911,0.091038,107.500000,109.480003,0.018419,1,range,0,,winner,0,NaN,,False
6,7,long,1,2022-10-25 00:00:00+00:00,2022-10-28 00:00:00+00:00,109.459999,113.838399,109.459999,113.838399,signal_entry,take_profit,10642.482632,10637.161390,11062.647846,11057.116522,5.321241,5.531324,10.852565,414.633890,0.03896,4,72.0,True,0,Tuesday,1,NaN,0.018907,0.133436,110.339996,113.699997,0.030451,1,range,0,,winner,0,NaN,,False
7,8,long,1,2022-10-31 00:00:00+00:00,2022-11-23 00:00:00+00:00,112.000000,116.480000,112.000000,116.480000,signal_entry,take_profit,11057.116522,11051.587964,11493.651482,11487.904657,5.528558,5.746826,11.275384,430.788135,0.03896,18,552.0,True,0,Monday,0,NaN,0.016771,0.229092,112.720001,116.379997,0.032470,1,range,0,,winner,0,NaN,,False
8,9,long,1,2022-11-24 00:00:00+00:00,2023-01-06 00:00:00+00:00,116.080002,120.723202,116.080002,120.723202,signal_entry,take_profit,11487.904657,11482.160704,11941.447133,11935.476409,5.743952,5.970724,11.714676,447.571752,0.03896,31,1032.0,True,0,Thursday,3,NaN,0.011741,0.076752,116.440002,122.400002,0.051185,1,range,0,,winner,0,NaN,,False
9,10,long,1,2023-01-09 00:00:00+00:00,2023-01-13 00:00:00+00:00,122.540001,127.441601,122.540001,127.441601,signal_entry,take_profit,11935.476409,11929.508671,12406.689018,12400.485673,5.967738,6.203345,12.171083,465.009264,0.03896,5,96.0,True,0,Monday,0,NaN,0.011129,0.047513,122.139999,126.540001,0.036024,1,trend,0,,winner,0,NaN,,False


...

[viz] report_v2


[viz] report_v3


,check,passed,details
0,1.base_regression,False,
1,2.artifact_root_stable,True,/Users/loic/Documents/Code/backtester-dash/res...
2,3.dashboard_comparison,True,dashboard/comparison/report figs present
3,4.trade_enriched,True,trade enrichment valid
4,5.split_metrics,True,train/val/test metrics present
5,6.hmm_selection_regimes,True,hmm selection + regimes present
6,7.scoring_quality,True,"ok, feature_stability_rows=10"
7,8.robustness_scenarios,True,"walkforward=6, stress=16"
8,9.failure_diagnosis,True,rows=5
9,10.stat_robustness,True,"summary_rows=1, samples=500"


Extended path: /Users/loic/Documents/Code/DL/Artificial-Neural-Network-based-Stock-Trading-System/research/runs/20260412T201542Z_saf.pa_1d


,run_id,created_at_utc,run_path,symbol,timeframe,start,end,strategy_tag,config_hash,raw_market_hash,train_hash,total_pnl,cumulative_return,sharpe_ratio,max_drawdown,win_rate,profit_factor,total_trades,notes
11,20260412T200231Z_en.pa_1d,2026-04-12 20:02:31+00:00,/Users/loic/Documents/Code/DL/Artificial-Neura...,EN.PA,1d,2022-07-14T00:00:00Z,2026-03-09T00:00:00Z,ema20_50_sl0.06_tp0.06,bae2d47bd83deab4ff93f91948f270d3f0a7972ebafc17...,588b18ccc526d7d862b674733d74c0ab7a420a7df902d2...,588b18ccc526d7d862b674733d74c0ab7a420a7df902d2...,9070.795972,0.907080,5.983160,-0.134759,0.651163,1.732514,43,ANN_long_short -> backtest_lib external pipeline
12,20260412T200241Z_en.pa_1d,2026-04-12 20:02:42+00:00,/Users/loic/Documents/Code/DL/Artificial-Neura...,EN.PA,1d,2022-07-14T00:00:00Z,2026-03-09T00:00:00Z,ema20_50_sl0.06_tp0.06,bae2d47bd83deab4ff93f91948f270d3f0a7972ebafc17...,588b18ccc526d7d862b674733d74c0ab7a420a7df902d2...,588b18ccc526d7d862b674733d74c0ab7a420a7df902d2...,9070.795972,0.907080,5.983160,-0.134759,0.651163,1.732514,43,ANN_long_short -> backtest_lib external pipeline
13,20260412T200706Z_en.pa_1d,2026-04-12 20:07:06+00:00,/Users/loic/Documents/Code/DL/Artificial-Neura...,EN.PA,1d,2022-07-14T00:00:00Z,2026-03-09T00:00:00Z,ema20_50_sl0.08_tp0.08,fbf46edf33042464c0b710aedb88157494e59f8b929dd0...,588b18ccc526d7d862b674733d74c0ab7a420a7df902d2...,588b18ccc526d7d862b674733d74c0ab7a420a7df902d2...,8695.529893,0.869553,5.647073,-0.177055,0.692308,2.084348,26,ANN_long_short -> backtest_lib external pipeline
14,20260412T200715Z_en.pa_1d,2026-04-12 20:07:15+00:00,/Users/loic/Documents/Code/DL/Artificial-Neura...,EN.PA,1d,2022-07-14T00:00:00Z,2026-03-09T00:00:00Z,ema20_50_sl0.08_tp0.08,fbf46edf33042464c0b710aedb88157494e59f8b929dd0...,588b18ccc526d7d862b674733d74c0ab7a420a7df902d2...,588b18ccc526d7d862b674733d74c0ab7a420a7df902d2...,8695.529893,0.869553,5.647073,-0.177055,0.692308,2.084348,26,ANN_long_short -> backtest_lib external pipeline
15,20260412T201157Z_enx.pa_1d,2026-04-12 20:11:57+00:00,/Users/loic/Documents/Code/DL/Artificial-Neura...,ENX.PA,1d,2024-07-08T00:00:00Z,2026-03-09T00:00:00Z,ema20_50_sl0.08_tp0.08,76d560ecf94cac2b4328e3c93f415f5a83b3be1a49f187...,0868eb969ceee1bec28977b57a6d4472197baee61348df...,0868eb969ceee1bec28977b57a6d4472197baee61348df...,0.000000,0.000000,NaN,0.000000,NaN,NaN,0,ANN_long_short -> backtest_lib external pipeline
16,20260412T201202Z_enx.pa_1d,2026-04-12 20:12:03+00:00,/Users/loic/Documents/Code/DL/Artificial-Neura...,ENX.PA,1d,2024-07-08T00:00:00Z,2026-03-09T00:00:00Z,ema20_50_sl0.08_tp0.08,76d560ecf94cac2b4328e3c93f415f5a83b3be1a49f187...,0868eb969ceee1bec28977b57a6d4472197baee61348df...,0868eb969ceee1bec28977b57a6d4472197baee61348df...,0.000000,0.000000,NaN,0.000000,NaN,NaN,0,ANN_long_short -> backtest_lib external pipeline
17,20260412T201320Z_saf.pa_1d,2026-04-12 20:13:20+00:00,/Users/loic/Documents/Code/DL/Artificial-Neura...,SAF.PA,1d,2022-08-09T00:00:00Z,2026-03-09T00:00:00Z,ema20_50_sl0.08_tp0.08,a1dcfbacba5688ad30bcfb50caa904147e59b315dc89d1...,d3fc0d6c84d56d67d831a13681e7df7b2f4385ebf42467...,d3fc0d6c84d56d67d831a13681e7df7b2f4385ebf42467...,5645.734000,0.564573,3.890727,-0.327536,0.636364,1.504867,33,ANN_long_short -> backtest_lib external pipeline
18,20260412T201330Z_saf.pa_1d,2026-04-12 20:13:30+00:00,/Users/loic/Documents/Code/DL/Artificial-Neura...,SAF.PA,1d,2022-08-09T00:00:00Z,2026-03-09T00:00:00Z,ema20_50_sl0.08_tp0.08,a1dcfbacba5688ad30bcfb50caa904147e59b315dc89d1...,d3fc0d6c84d56d67d831a13681e7df7b2f4385ebf42467...,d3fc0d6c84d56d67d831a13681e7df7b2f4385ebf42467...,5645.734000,0.564573,3.890727,-0.327536,0.636364,1.504867,33,ANN_long_short -> backtest_lib external pipeline
19,20260412T201533Z_saf.pa_1d,2026-04-12 20:15:33+00:00,/Users/loic/Documents/Code/DL/Artificial-Neura...,SAF.PA,1d,2022-08-09T00:00:00Z,2026-03-09T00:00:00Z,ema20_50_sl0.12_tp0.04,980596bae084d5194c403c97fb9761db8deb19a00c219c...,d3fc0d6c84d56d67d831a13681e7df7b2f4385ebf42467...,d3fc0d6c84d56d67d831a13681e7df7b